# Lab 1.5 — C# Unit Test Generation with Gemini

This notebook uses Gemini to generate xUnit tests for the C# OrderQuantityValidator. It compares zero-shot and few-shot prompts using the same source code as input.

In [2]:
%pip install "google-genai==2.12.1" "google-auth==2.49.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 9.8 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.63.1
    Uninstalling google-auth-2.63.1:
      Successfully uninstalled google-auth-2.63.1
  Attempting uninstall: google-genai
    Found existing installation: google-genai 3.6.1
    Uninstalling google-genai-3.6.1:
      Successfully uninstalled google-genai-3.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [google-genai]


In [6]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_KEY"))
model = "gemini-3.5-flash-lite"

print("Gemini client initialized.")

Gemini client initialized.


In [4]:
from urllib.request import urlopen

base_url = (
    "https://raw.githubusercontent.com/Thuthutka/"
    "Intelektualios-informacines-sistemos/main/prompts"
)

prompts = {}
for mode in ("zero_shot", "few_shot"):
    with urlopen(f"{base_url}/{mode}.md", timeout=30) as response:
        prompts[mode] = response.read().decode("utf-8")

marker = "## New observation (X)"
zero_input = prompts["zero_shot"].split(marker, 1)[1].strip()
few_input = prompts["few_shot"].split(marker, 1)[1].strip()

assert zero_input == few_input, "The prompts use different inputs."

print("Both prompts loaded. Shared input X:\n")
print(zero_input)

Both prompts loaded. Shared input X:

Test framework: xUnit.
Production namespace: Lab14.Target.
Expected behavior: An order quantity is allowed if it is positive, does not exceed available stock, and is at most 10 items per order. A quantity equal to either upper limit is allowed.

```csharp
namespace Lab14.Target;

public static class OrderQuantityValidator
{
    public const int MaxPerOrder = 10;

    public static bool IsAllowed(int quantity, int availableStock)
    {
        return quantity > 0
            && quantity <= availableStock
            && quantity <= MaxPerOrder;
    }
}
```

Generate y for the new observation.


In [5]:
results = {}

for mode, prompt in prompts.items():
    print(f"\n=== {mode} ===")
    print("Reference examples:", 0 if mode == "zero_shot" else 3)
    print("Model:", model)

    response = client.models.generate_content(
        model=model,
        contents=prompt,
    )

    results[mode] = (response.text or "").strip()
    if not results[mode]:
        raise RuntimeError(f"Gemini returned no text for {mode}.")

    print("\nOutput y — generated xUnit tests\n")
    print(results[mode])


=== zero_shot ===
Reference examples: 0
Model: gemini-3.5-flash-lite

Output y — generated xUnit tests

using Xunit;
namespace Lab14.Target.Tests;

public class OrderQuantityValidatorTests
{
    [Theory]
    [InlineData(1, 10)]
    [InlineData(5, 5)]
    [InlineData(10, 10)]
    [InlineData(10, 20)]
    public void IsAllowed_ValidQuantities_ReturnsTrue(int quantity, int availableStock)
    {
        bool result = OrderQuantityValidator.IsAllowed(quantity, availableStock);
        Assert.True(result);
    }

    [Theory]
    [InlineData(0, 10)]
    [InlineData(-1, 10)]
    [InlineData(11, 20)]
    [InlineData(5, 4)]
    [InlineData(1, 0)]
    public void IsAllowed_InvalidQuantities_ReturnsFalse(int quantity, int availableStock)
    {
        bool result = OrderQuantityValidator.IsAllowed(quantity, availableStock);
        Assert.False(result);
    }
}

=== few_shot ===
Reference examples: 3
Model: gemini-3.5-flash-lite

Output y — generated xUnit tests

using Lab14.Target;
using Xunit;

## Results comparison

Both prompts used the same `OrderQuantityValidator` input and the same Gemini
model. The zero-shot prompt generated nine test cases, while the few-shot
prompt generated eight.

| Scenario | Zero-shot | Few-shot |
| --- | --- | --- |
| Reject zero and negative quantities | Yes | Yes |
| Accept a quantity equal to stock below the per-order limit | Yes `(5, 5)` | No |
| Reject a quantity above available stock | Yes | Yes |
| Accept 10 items when more than 10 are in stock | Yes `(10, 20)` | Yes `(10, 15)` |
| Reject 11 items when stock is sufficient | Yes | Yes |
| Reject a positive quantity when stock is zero | Yes `(1, 0)` | No |

The zero-shot result used separate theories for valid and invalid inputs.
The few-shot result used one theory with an explicit expected result for
each case, following the pattern in the reference examples. In this run,
zero-shot covered more boundary scenarios. The comparison is based on
inspection of the generated code.